<a href="https://colab.research.google.com/github/lctung/fontdiffuser-finetune-colab/blob/main/FontDiffuser_sample.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**下載Fontdiffuser專案**:

---


In [ ]:
!git clone https://github.com/lctung/fontdiffuser-finetune.git
%cd fontdiffuser-finetune
import os
os.rename("characters.txt", "V.txt")

**下載torch相關函式庫:**

---


# **下面這格程式執行中會需要按一次enter**

In [ ]:
# 安裝 Python 3.10
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-distutils python3.10-dev -y

# 切換默認 python 版本
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.12 1
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 2
!sudo update-alternatives --config python3

# 重新安裝 pip
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10

In [ ]:
!pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu117

In [ ]:
import torch
print("GPU 可用:", torch.cuda.is_available())
print("PyTorch 版本:", torch.__version__)

**安裝所需套件**:

---

In [ ]:
!pip install -r requirements.txt

**載入微調權重檔**

---



In [ ]:
# 切換工作目錄
%cd /content/fontdiffuser-finetune

# 安裝 gdown（如果尚未安裝）
!pip install -U gdown

# 建立 ckpt 資料夾（如果尚未存在）
!mkdir -p ckpt

# 下載 scr_210000.pth 檔案
!gdown --id 1UF4nIcL3PRJeQOQOPFFuGzj30v67rlNn -O ckpt/scr_210000.pth

# 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 複製三個 checkpoint 檔案到 /content/fontdiffuser-finetune/ckpt
import shutil
import os

# 設定來源與目標路徑
source_folder = '/content/drive/MyDrive/Fontdiffuser_finetuning_ckpt/global_step_5000'
target_folder = '/content/fontdiffuser-finetune/ckpt'

# 建立目標資料夾（若不存在）
os.makedirs(target_folder, exist_ok=True)

# 要複製的檔案列表
files_to_copy = ['content_encoder.pth', 'style_encoder.pth', 'unet.pth']

# 複製檔案
for file_name in files_to_copy:
    shutil.copy(os.path.join(source_folder, file_name), os.path.join(target_folder, file_name))

print("✅ 所有檔案下載與複製完成")


**上傳風格參考圖(從訓練集挑一張圖上傳)**

---



In [ ]:
from google.colab import files
import os
import shutil

# 1. 使用者上傳一張圖片
uploaded = files.upload()

# 2. 取得第一張圖片的檔名與副檔名
filename = next(iter(uploaded))  # 取第一個 key（檔名）
ext = os.path.splitext(filename)[1]  # 副檔名，例如 '.jpg'
target_dir = "/content/fontdiffuser-finetune/data_examples/sampling"

# 3. 重新命名並移動
new_filename = f"example_style.jpg"
target_path = os.path.join(target_dir, new_filename)
shutil.move(filename, target_path)

print(f"✅ 檔案已上傳並移動到：{target_path}")

## ⚠️ 開始生成前提醒：請先打開 V.txt，確認裡面的字是不是你想要生成的字體內，有需要更改的話，可以直接編輯這個檔案，把要生成的字輸入進去。

**開始進行字體生成**:

---

In [ ]:
%cd /content/fontdiffuser-finetune
!python sample.py \
  --ckpt_dir="ckpt/" \
  --style_image_path="data_examples/sampling/example_style.jpg" \
  --save_image \
  --character_input \
  --character_list_path="V.txt" \
  --save_image_dir="outputs_generater_character_list/" \
  --device="cuda:0" \
  --algorithm_type="dpmsolver++" \
  --guidance_type="classifier-free" \
  --guidance_scale=7.5 \
  --num_inference_steps=20 \
  --method="multistep"


**修改圖檔大小**

---



In [ ]:
from PIL import Image
import os
from pathlib import Path

# 設定圖片資料夾路徑
image_folder = 'outputs_generater_character_list'  # 請將此處替換為您的資料夾路徑

# 取得所有 PNG 檔案
png_files = list(Path(image_folder).glob('*.png'))

for file_path in png_files:
    try:
        with Image.open(file_path) as img:
            # 調整圖片大小為 300x300
            img_resized = img.resize((300, 300), Image.LANCZOS)
            # 儲存並覆蓋原始檔案
            img_resized.save(file_path)
            print(f"已處理：{file_path.name}")
    except Exception as e:
        print(f"處理 {file_path.name} 時發生錯誤：{e}")


**壓縮生成圖片檔案並下載**

---



In [ ]:
!zip -r outputs_generater_character_list.zip outputs_generater_character_list
from google.colab import files
files.download('outputs_generater_character_list.zip')